# Step 0. 실행 환경 구축

Qwen3.5-122B-A10B-GPTQ-Int4 모델을 A100 80GB 단일 GPU에서
vLLM으로 실행하기 위한 환경을 구성한다.

검증 완료된 실행 환경:

- Python 3.13.x
- PyTorch 2.13.0+cu132
- CUDA 13.2
- TorchVision 0.28.0+cu132
- vLLM 0.29.1rc1.dev567+g8d16ca6cc
- GPU: NVIDIA A100-SXM4-80GB

이 단계에서는 모델과 데이터를 사용하지 않고,
122B 모델을 안정적으로 실행할 수 있는 라이브러리 환경만 구성한다.

설치 완료 후 런타임을 한 번 재시작하고 Step 1을 진행한다.

In [8]:
# ============================================================
# [STEP 0] QWEN3.5-122B WORKING ENVIRONMENT
#
# 실제 성공 환경 재현:
# Python      : 3.13.x
# Torch       : 2.13.0+cu132
# CUDA        : 13.2
# TorchVision : 0.28.0+cu132
# vLLM        : 0.29.1rc1.dev567+g8d16ca6cc
# TorchAudio  : NOT INSTALLED
#
# 이번 셀에서:
# 1. Torch / TorchVision CU132 설치
# 2. vLLM dev567 설치
# 3. vLLM이 요구하는 dependencies 전부 자동 설치
# 4. cbor2 / openai-harmony / msgspec 포함
# 5. fresh subprocess에서 from vllm import LLM 검증
#
# 모델 다운로드 X
# 데이터 사용 X
#
# 완료 후 세션 다시 시작 1회
# ============================================================

import sys
import subprocess
import importlib.metadata as md


# ============================================================
# CONFIG
# ============================================================

VLLM_COMMIT = (
    "8d16ca6cc21931481f6c6f9ec0365874a80c6bb2"
)

VLLM_VERSION = (
    "0.29.1rc1.dev567+g8d16ca6cc"
)

VLLM_INDEX = (
    f"https://wheels.vllm.ai/{VLLM_COMMIT}"
)


print("=" * 72)
print("STEP 0 : BUILD EXACT 122B ENVIRONMENT")
print("=" * 72)


# ============================================================
# 1. REMOVE CONFLICTING PACKAGES
# ============================================================

print("\n===== [0-0] REMOVE OLD STACK =====")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torch",
        "torchvision",
        "torchaudio",
        "vllm",
    ],
    check=False,
)


# ============================================================
# 2. TORCH 2.13 + CUDA 13.2
# ============================================================

print("\n===== [0-1] INSTALL TORCH CU132 =====")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--force-reinstall",

        "torch==2.13.0+cu132",
        "torchvision==0.28.0+cu132",

        "--index-url",
        "https://download.pytorch.org/whl/cu132",
    ],
    check=True,
)


# ============================================================
# 3. EXACT vLLM dev567
#
# 먼저 wheel 자체만 설치.
# 그 다음 metadata에서 dependency를 전부 읽어서 설치.
# ============================================================

print("\n===== [0-2] INSTALL EXACT vLLM dev567 =====")

subprocess.run(
    [
        "uv",
        "pip",
        "install",

        "--reinstall",
        "--no-deps",
        "--pre",

        "vllm",

        "--index-url",
        VLLM_INDEX,
    ],
    check=True,
)


# ============================================================
# 4. READ ALL vLLM DEPENDENCIES
#
# import vllm 하지 않음.
# package metadata만 읽기 때문에 안전.
# ============================================================

print("\n===== [0-3] RESOLVE ALL vLLM DEPENDENCIES =====")

from packaging.requirements import Requirement


requires = md.requires("vllm") or []

deps = []


for req_text in requires:

    try:
        req = Requirement(req_text)

    except Exception:
        continue


    # --------------------------------------------------------
    # 현재 Python/platform에서 marker가 false면 제외
    # --------------------------------------------------------

    if (
        req.marker is not None
        and not req.marker.evaluate()
    ):
        continue


    name = req.name.lower()


    # --------------------------------------------------------
    # 이미 우리가 정확한 wheel로 관리하는 것들
    # --------------------------------------------------------

    if name in {
        "torch",
        "torchvision",
        "torchaudio",
        "vllm",
    }:
        continue


    deps.append(
        req_text
    )


# 중복 제거
deps = list(
    dict.fromkeys(deps)
)


print(
    "vLLM dependencies:",
    len(deps)
)


# ============================================================
# 5. INSTALL ALL DEPENDENCIES
#
# 여기서:
# cbor2
# msgspec
# openai-harmony
# 등 dev567이 요구하는 패키지를 전부 설치
# ============================================================

print("\n===== [0-4] INSTALL ALL vLLM DEPENDENCIES =====")


if deps:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",

            "--upgrade",

            *deps,
        ],
        check=True,
    )


# ============================================================
# 6. TORCHAUDIO 제거 유지
#
# 성공 환경에서도 NOT INSTALLED
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchaudio",
    ],
    check=False,
)


# ============================================================
# 7. FRESH PROCESS VERIFY
#
# 현재 notebook kernel은 기존 module을 들고 있을 수 있으므로
# 새 Python process에서 실제 import 검증.
# ============================================================

print("\n===== [0-5] FRESH PROCESS VERIFY =====")


verify_code = r'''
import sys
import importlib.metadata as md

import torch
import torchvision
import vllm

from vllm import LLM
from vllm import SamplingParams
from vllm.sampling_params import StructuredOutputsParams


print("Python      :", sys.version.split()[0])
print("Torch       :", torch.__version__)
print("CUDA        :", torch.version.cuda)
print("TorchVision :", torchvision.__version__)
print("vLLM        :", vllm.__version__)

try:
    print("TorchAudio  :", md.version("torchaudio"))
except Exception:
    print("TorchAudio  : NOT INSTALLED")

print(
    "GPU         :",
    torch.cuda.get_device_name(0)
)

print(
    "VRAM        :",
    round(
        torch.cuda.get_device_properties(0)
        .total_memory / 1024**3,
        2,
    ),
    "GB",
)


assert torch.__version__.startswith(
    "2.13.0+cu132"
), torch.__version__

assert torch.version.cuda == "13.2"

assert torchvision.__version__.startswith(
    "0.28.0+cu132"
), torchvision.__version__

assert (
    "0.29.1rc1.dev567"
    in vllm.__version__
), vllm.__version__


print()
print("LLM IMPORT                     : OK")
print("SamplingParams IMPORT          : OK")
print("StructuredOutputsParams IMPORT : OK")

print()
print("=" * 60)
print("✅ STEP 0 ENVIRONMENT SUCCESS")
print("=" * 60)
'''


result = subprocess.run(
    [
        sys.executable,
        "-c",
        verify_code,
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)


print(
    result.stdout
)


if result.returncode != 0:

    raise RuntimeError(
        "❌ STEP 0 VERIFY FAILED\n\n"
        + result.stdout
    )


# ============================================================
# DONE
# ============================================================

print("\n" + "=" * 72)

print("✅ STEP 0 COMPLETE")

print("=" * 72)

print(
    "\n이제 Colab 메뉴에서 딱 한 번:\n"
    "런타임 -> 세션 다시 시작\n\n"
    "연결 해제 X\n"
    "런타임 삭제 X\n"
    "data.zip 다시 풀기 X\n"
    "122B 다시 다운로드 X\n"
)

print("=" * 72)

STEP 0 : BUILD EXACT 122B ENVIRONMENT

===== [0-0] REMOVE OLD STACK =====

===== [0-1] INSTALL TORCH CU132 =====

===== [0-2] INSTALL EXACT vLLM dev567 =====

===== [0-3] RESOLVE ALL vLLM DEPENDENCIES =====
vLLM dependencies: 72

===== [0-4] INSTALL ALL vLLM DEPENDENCIES =====

===== [0-5] FRESH PROCESS VERIFY =====
Python      : 3.13.15
Torch       : 2.13.0+cu132
CUDA        : 13.2
TorchVision : 0.28.0+cu132
vLLM        : 0.29.1rc1.dev567+g8d16ca6cc
TorchAudio  : NOT INSTALLED
GPU         : NVIDIA A100-SXM4-80GB
VRAM        : 79.25 GB

LLM IMPORT                     : OK
SamplingParams IMPORT          : OK
StructuredOutputsParams IMPORT : OK

✅ STEP 0 ENVIRONMENT SUCCESS


✅ STEP 0 COMPLETE

이제 Colab 메뉴에서 딱 한 번:
런타임 -> 세션 다시 시작

연결 해제 X
런타임 삭제 X
data.zip 다시 풀기 X
122B 다시 다운로드 X



# Step 1. Qwen3.5-122B 모델 준비 및 GPU 로드

Qwen3.5-122B-A10B-GPTQ-Int4 모델을 준비하고
A100 80GB 단일 GPU에 vLLM으로 로드한다.

모델이 로컬에 존재하지 않으면 Hugging Face에서 다운로드하고,
이미 정상적으로 다운로드되어 있으면 기존 파일을 재사용한다.

검증된 모델 로드 설정:

- model: Qwen/Qwen3.5-122B-A10B-GPTQ-Int4
- quantization: moe_wna16
- dtype: bfloat16
- tensor_parallel_size: 1
- max_model_len: 2048
- max_num_seqs: 1
- gpu_memory_utilization: 0.99
- enforce_eager: True
- multimodal image input: 1
- trust_remote_code: True
- seed: 42

모델 로드가 완료되면 `llm_122b` 객체를 유지한 상태로
이후 Image-only VQA 추론을 진행한다.

In [1]:
# ============================================================
# [STEP 1] QWEN3.5-122B PREPARE + LOAD
#
# 동작:
# 1. 로컬 모델이 없거나 불완전하면 snapshot_download
# 2. 이미 완전하면 다운로드 생략
# 3. llm_122b가 이미 메모리에 있으면 재로드 생략
# 4. 새 세션이면 122B GPU 로드
#
# 따라서:
# - 지금 실행해도 안전
# - 새 런타임에서도 단독 실행 가능
# ============================================================

import time
import shutil
from pathlib import Path

import torch
import torchvision
import vllm

from huggingface_hub import snapshot_download
from vllm import LLM


# ============================================================
# 0. CONFIG
# ============================================================

MODEL_ID = "Qwen/Qwen3.5-122B-A10B-GPTQ-Int4"

MODEL_DIR = Path(
    "/content/Qwen3.5-122B-A10B-GPTQ-Int4"
)


# ============================================================
# 1. ENV VERIFY
# ============================================================

print("=" * 72)
print("STEP 1 : QWEN3.5-122B PREPARE + LOAD")
print("=" * 72)

print("Torch       :", torch.__version__)
print("CUDA        :", torch.version.cuda)
print("TorchVision :", torchvision.__version__)
print("vLLM        :", vllm.__version__)
print("GPU         :", torch.cuda.get_device_name(0))

print(
    "VRAM        :",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2,
    ),
    "GB",
)


assert torch.__version__.startswith(
    "2.13.0+cu132"
), torch.__version__

assert torch.version.cuda == "13.2"

assert torchvision.__version__.startswith(
    "0.28.0+cu132"
), torchvision.__version__

assert "0.29.1rc1.dev567" in vllm.__version__


print("\n✅ ENVIRONMENT VERIFIED")


# ============================================================
# 2. MODEL FILE CHECK
# ============================================================

def model_is_complete():

    if not MODEL_DIR.exists():
        return False

    if not (MODEL_DIR / "config.json").exists():
        return False

    weights = list(
        MODEL_DIR.glob("*.safetensors")
    )

    if not weights:
        return False

    total_gb = sum(
        p.stat().st_size
        for p in weights
    ) / 1024**3

    # 현재 정상 다운로드 결과 약 73.45GB
    if total_gb < 70:
        return False

    return True


# ============================================================
# 3. DOWNLOAD IF NEEDED
# ============================================================

if model_is_complete():

    weights = list(
        MODEL_DIR.glob("*.safetensors")
    )

    total_gb = sum(
        p.stat().st_size
        for p in weights
    ) / 1024**3

    print("\n===== MODEL CHECK =====")
    print("✅ EXISTING MODEL READY")
    print("MODEL DIR   :", MODEL_DIR)
    print("WEIGHT FILES:", len(weights))
    print("WEIGHT SIZE :", round(total_gb, 2), "GB")

else:

    free_gb = (
        shutil.disk_usage("/content").free
        / 1024**3
    )

    print("\n===== MODEL DOWNLOAD =====")
    print("DISK FREE:", round(free_gb, 2), "GB")

    assert free_gb > 85, (
        f"❌ 모델 다운로드 공간 부족: {free_gb:.2f}GB"
    )

    t0 = time.time()

    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_DIR),
        max_workers=8,
    )

    print(
        "✅ DOWNLOAD COMPLETE:",
        round(
            (time.time() - t0) / 60,
            2,
        ),
        "min",
    )


# ============================================================
# 4. FINAL FILE VERIFY
# ============================================================

assert model_is_complete(), (
    "❌ 모델 다운로드/파일 검증 실패"
)

weights = list(
    MODEL_DIR.glob("*.safetensors")
)

total_gb = sum(
    p.stat().st_size
    for p in weights
) / 1024**3


print("\n===== MODEL READY =====")
print("WEIGHT FILES:", len(weights))
print("WEIGHT SIZE :", round(total_gb, 2), "GB")


# ============================================================
# 5. LOAD OR REUSE
# ============================================================

if (
    "llm_122b" in globals()
    and llm_122b is not None
):

    print("\n" + "=" * 72)
    print("✅ EXISTING llm_122b REUSED")
    print("OBJECT:", type(llm_122b))
    print("=" * 72)

else:

    print("\n===== START 122B LOAD =====")

    t0 = time.time()

    llm_122b = LLM(

        model=str(MODEL_DIR),

        tensor_parallel_size=1,

        quantization="moe_wna16",

        dtype="bfloat16",

        max_model_len=2048,

        max_num_seqs=1,

        gpu_memory_utilization=0.99,

        enforce_eager=True,

        limit_mm_per_prompt={
            "image": 1,
            "video": 0,
        },

        trust_remote_code=True,

        seed=42,
    )

    elapsed = time.time() - t0

    print("\n" + "=" * 72)
    print("✅ STEP 1 122B LOAD SUCCESS")
    print("=" * 72)

    print(
        "LOAD TIME:",
        round(elapsed / 60, 2),
        "min",
    )

    print(
        "OBJECT:",
        type(llm_122b),
    )

    print("=" * 72)

STEP 1 : QWEN3.5-122B PREPARE + LOAD
Torch       : 2.13.0+cu132
CUDA        : 13.2
TorchVision : 0.28.0+cu132
vLLM        : 0.29.1rc1.dev567+g8d16ca6cc
GPU         : NVIDIA A100-SXM4-80GB
VRAM        : 79.25 GB

✅ ENVIRONMENT VERIFIED

===== MODEL CHECK =====
✅ EXISTING MODEL READY
MODEL DIR   : /content/Qwen3.5-122B-A10B-GPTQ-Int4
WEIGHT FILES: 39
WEIGHT SIZE : 73.45 GB

===== MODEL READY =====
WEIGHT FILES: 39
WEIGHT SIZE : 73.45 GB

===== START 122B LOAD =====
INFO 09-25 06:15:26 [api_utils.py:286] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'seed': 42, 'max_model_len': 2048, 'gpu_memory_utilization': 0.99, 'max_num_seqs': 1, 'disable_log_stats': True, 'quantization': 'moe_wna16', 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 1, 'video': 0}, 'model': '/content/Qwen3.5-122B-A10B-GPTQ-Int4'}
INFO 09-25 06:15:26 [model.py:696] Resolved architecture: Qwen3_5MoeForConditionalGeneration
INFO 09-25 06:15:26 [model.py:2104] Using max model len 2048
WARNING 0

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


WARNING 09-25 06:15:45 [system_utils.py:156] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 09-25 06:16:11 [base.py:251] Multi-modal warmup completed in 26.418s
INFO 09-25 06:19:04 [hf.py:642] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.

✅ STEP 1 122B LOAD SUCCESS
LOAD TIME: 3.64 min
OBJECT: <class 'vllm.entrypoints.llm.LLM'>


# Step 2. Image-only 성능 검증

Qwen3.5-122B 모델의 VQA 성능을 확인하기 위해
정답이 존재하는 clean train 데이터 중 앞 1,000개를 대상으로 검증한다.

이전 실험에서는 OCR 및 Object Detection 정보를 프롬프트에 추가했지만,
오히려 모델의 이미지 이해를 방해하며 정확도가 하락했다.

따라서 최종 전략에서는 별도의 OCR/DINO 정보를 사용하지 않고
다음 정보만 122B 모델에 입력한다.

- 실제 이미지
- 질문
- 선택지 a / b / c / d

추론 설정:

- enable_thinking=False
- temperature=0.0
- structured output: a / b / c / d
- max_tokens=2

1,000개 Image-only 검증 결과:

**984 / 1000 = 98.4%**

따라서 이후 Test 데이터 추론에서도
동일한 Image-only 설정을 기본 추론 방식으로 사용한다.

In [3]:
# ============================================================
# [STEP 2-3 PREP] IMAGE-ONLY 데이터 환경 복구
#
# 세션 재시작 후:
# - Drive mount
# - clean CSV 다시 로드
# - DATA_ROOT 복구
# - 깨진 image-only 결과 파일 제거
# - 1000개 이미지 존재 확인
# ============================================================

from pathlib import Path

import pandas as pd
from google.colab import drive


# ============================================================
# 0. MODEL CHECK
# ============================================================

assert (
    "llm_122b" in globals()
    and llm_122b is not None
), "❌ Step 1 모델 로드부터 필요합니다."

print("✅ llm_122b:", type(llm_122b))


# ============================================================
# 1. DRIVE
# ============================================================

drive.mount(
    "/content/drive"
)


# ============================================================
# 2. PATH
# ============================================================

CLEAN_CSV = Path(
    "/content/drive/MyDrive/"
    "cleaning_data/"
    "train_combined_clean.csv"
)

DATA_ROOT = Path(
    "/content/ssafy_data"
)

OUT_CSV = Path(if "status" in old.columns:
    "/content/drive/MyDrive/"
    "cleaning_data/"
    "122b_eval/"
    "122b_image_only_1000.csv"
)


assert CLEAN_CSV.exists(), (
    f"❌ CSV 없음: {CLEAN_CSV}"
)

assert DATA_ROOT.exists(), (
    f"❌ DATA_ROOT 없음: {DATA_ROOT}"
)


# ============================================================
# 3. CSV RELOAD
# ============================================================

df = pd.read_csv(
    CLEAN_CSV
)


required_cols = [
    "id",
    "path",
    "question",
    "a",
    "b",
    "c",
    "d",
    "answer",
]


missing = [
    c
    for c in required_cols
    if c not in df.columns
]


assert not missing, (
    f"❌ 컬럼 누락: {missing}"
)


df["answer"] = (
    df["answer"]
    .astype(str)
    .str.strip()
    .str.lower()
)


df["source"] = (
    df["path"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
    .str.split("/")
    .str[0]
    .str.lower()
)


# ============================================================
# 4. VERIFY
# ============================================================

print("\n===== CLEAN DATA =====")

print(
    "ROWS:",
    len(df)
)

print(
    "SOURCE:",
    df["source"]
    .value_counts()
    .to_dict()
)


assert len(df) == 7169, (
    f"❌ 예상 7169행, 실제 {len(df)}행"
)


first_1000_exists = df.iloc[:1000]["path"].map(
    lambda p:
    (DATA_ROOT / str(p)).exists()
)


print(
    "FIRST 1000 IMAGES:",
    int(first_1000_exists.sum()),
    "/ 1000"
)


assert first_1000_exists.all(), (
    "❌ 첫 1000개 이미지 중 누락 있음"
)


# ============================================================
# 6. READY
# ============================================================

print("\n" + "=" * 72)
print("✅ STEP 2-3 PREP COMPLETE")
print("=" * 72)

print("df        :", len(df))
print("DATA_ROOT :", DATA_ROOT)
print("MODEL     :", type(llm_122b))
print("=" * 72)

✅ llm_122b: <class 'vllm.entrypoints.llm.LLM'>
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

===== CLEAN DATA =====
ROWS: 7169
SOURCE: {'train': 5527, 'dev': 1642}
FIRST 1000 IMAGES: 1000 / 1000

기존 image-only 파일: 203 rows
{'error': 203}
✅ EngineDead 결과 파일 삭제

✅ STEP 2-3 PREP COMPLETE
df        : 7169
DATA_ROOT : /content/ssafy_data
MODEL     : <class 'vllm.entrypoints.llm.LLM'>


In [6]:
# ============================================================
# [STEP 2-3] IMAGE-ONLY 1000 SAMPLE SMOKE TEST
#
# 같은 clean dataset의 앞 1000개를 사용
#
# OCR X
# DINO X
#
# 실제 이미지 + question + a/b/c/d 만 사용
#
# 기존 OCR/DINO 결과와 별도 저장
# ============================================================

import os
import time
import inspect
import pandas as pd

from pathlib import Path
from PIL import Image

from vllm import SamplingParams
from vllm.sampling_params import StructuredOutputsParams


# ============================================================
# 0. CONFIG
# ============================================================

SMOKE_N = 1000

OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "cleaning_data/"
    "122b_eval"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUT_CSV = (
    OUT_DIR
    / "122b_image_only_1000.csv"
)


assert (
    "llm_122b" in globals()
    and llm_122b is not None
)

assert len(df) >= SMOKE_N


# ============================================================
# 1. EXACT SAME 1000 ROWS
# ============================================================

smoke_df = (
    df
    .iloc[:SMOKE_N]
    .copy()
    .reset_index(drop=True)
)


if "source" not in smoke_df.columns:

    smoke_df["source"] = (
        smoke_df["path"]
        .astype(str)
        .str.replace("\\", "/", regex=False)
        .str.split("/")
        .str[0]
        .str.lower()
    )


smoke_df["answer"] = (
    smoke_df["answer"]
    .astype(str)
    .str.strip()
    .str.lower()
)


print("=" * 78)
print("122B IMAGE-ONLY 1000 SMOKE TEST")
print("=" * 78)

print("SAMPLES :", len(smoke_df))
print("OUTPUT  :", OUT_CSV)


# ============================================================
# 2. IMAGE-ONLY SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = (
    "당신은 이미지 기반 4지선다 VQA 전문가입니다.\n"
    "이미지를 가장 중요한 근거로 사용하고 "
    "질문과 선택지를 신중하게 비교하세요.\n"
    "최종 답은 반드시 a, b, c, d 중 "
    "소문자 한 글자로만 출력하세요."
)


def build_image_only_prompt(
    question,
    a,
    b,
    c,
    d,
):

    return (
        f"질문: {question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "정답:"
    )


# ============================================================
# 3. STRUCTURED OUTPUT
# ============================================================

choice_constraint = StructuredOutputsParams(
    choice=[
        "a",
        "b",
        "c",
        "d",
    ]
)


sampling_params_image_only = SamplingParams(
    temperature=0.0,
    max_tokens=2,
    structured_outputs=choice_constraint,
)


# ============================================================
# 4. RESUME
# ============================================================

results = {}


if OUT_CSV.exists():

    old = pd.read_csv(
        OUT_CSV
    )

    for rec in (
        old
        .drop_duplicates(
            "id",
            keep="last"
        )
        .to_dict("records")
    ):

        results[
            str(rec["id"])
        ] = rec


done_ids = {
    k
    for k, v in results.items()
    if str(
        v.get(
            "status",
            ""
        )
    ) == "ok"
}


print(
    "RESUME  :",
    len(done_ids),
    "/",
    SMOKE_N
)


# ============================================================
# 5. SAVE
# ============================================================

def save_image_only():

    x = pd.DataFrame(
        list(
            results.values()
        )
    )

    tmp = Path(
        str(OUT_CSV)
        + ".tmp"
    )

    x.to_csv(
        tmp,
        index=False,
        encoding="utf-8-sig",
    )

    os.replace(
        tmp,
        OUT_CSV,
    )


# ============================================================
# 6. TQDM SUPPORT
# ============================================================

SUPPORTS_USE_TQDM = (
    "use_tqdm"
    in inspect.signature(
        llm_122b.chat
    ).parameters
)


# ============================================================
# 7. LOOP
# ============================================================

for idx, row in smoke_df.iterrows():

    img_id = str(
        row["id"]
    )


    if img_id in done_ids:
        continue


    gt = (
        str(
            row["answer"]
        )
        .strip()
        .lower()
    )


    image_path = (
        DATA_ROOT
        / str(
            row["path"]
        )
    )


    t0 = time.time()


    try:

        assert image_path.exists()


        with Image.open(
            image_path
        ) as im:

            image = (
                im
                .convert("RGB")
                .copy()
            )


        messages = [

            {
                "role":
                    "system",

                "content":
                    SYSTEM_PROMPT,
            },

            {
                "role":
                    "user",

                "content": [

                    {
                        "type":
                            "image_pil",

                        "image_pil":
                            image,
                    },

                    {
                        "type":
                            "text",

                        "text":
                            build_image_only_prompt(

                                row["question"],

                                row["a"],
                                row["b"],
                                row["c"],
                                row["d"],
                            ),
                    },
                ],
            },
        ]


        kwargs = dict(

            messages=
                messages,

            sampling_params=
                sampling_params_image_only,

            chat_template_kwargs={
                "enable_thinking":
                    False
            },
        )


        if SUPPORTS_USE_TQDM:

            kwargs[
                "use_tqdm"
            ] = False


        outputs = llm_122b.chat(
            **kwargs
        )


        pred = (
            outputs[0]
            .outputs[0]
            .text
            .strip()
            .lower()
        )


        assert pred in {
            "a",
            "b",
            "c",
            "d",
        }


        correct = int(
            pred == gt
        )


        status = "ok"
        error = ""


    except Exception as e:

        pred = ""
        correct = 0
        status = "error"
        error = repr(e)


    elapsed = (
        time.time()
        - t0
    )


    results[
        img_id
    ] = {

        "idx":
            int(idx),

        "id":
            img_id,

        "source":
            row["source"],

        "path":
            row["path"],

        "gt_answer":
            gt,

        "image_only_answer":
            pred,

        "correct":
            correct,

        "elapsed_sec":
            round(
                elapsed,
                4
            ),

        "status":
            status,

        "error":
            error,
    }


    if status == "ok":

        done_ids.add(
            img_id
        )


    save_image_only()


    current = pd.DataFrame(
        list(
            results.values()
        )
    )


    ok = current[
        current["status"]
        == "ok"
    ]


    correct_n = int(
        ok["correct"]
        .astype(int)
        .sum()
    )


    total_n = len(
        ok
    )


    acc = (
        correct_n / total_n
        if total_n
        else 0
    )


    mark = (
        "✅"
        if correct
        else "❌"
    )


    print(

        f"[{total_n:04d}/{SMOKE_N}] "

        f"{img_id} | "

        f"PRED={pred} "
        f"GT={gt} "

        f"{mark} | "

        f"ACC="
        f"{correct_n}/{total_n}="
        f"{acc:.4f} | "

        f"{elapsed:.2f}s",

        flush=True,
    )


# ============================================================
# 8. FINAL
# ============================================================

final = pd.DataFrame(
    list(
        results.values()
    )
)


ok = final[
    final["status"]
    == "ok"
]


correct_n = int(
    ok["correct"]
    .astype(int)
    .sum()
)


total_n = len(
    ok
)


acc = (
    correct_n / total_n
    if total_n
    else 0
)


print("\n" + "=" * 78)

print(
    "✅ IMAGE-ONLY 1000 COMPLETE"
)

print("=" * 78)

print(
    f"ACCURACY: "
    f"{correct_n}/{total_n} "
    f"= {acc:.6f}"
)

print(
    "RESULT:",
    OUT_CSV
)

print("=" * 78)

122B IMAGE-ONLY 1000 SMOKE TEST
SAMPLES : 1000
OUTPUT  : /content/drive/MyDrive/cleaning_data/122b_eval/122b_image_only_1000.csv
RESUME  : 0 / 1000
[0001/1000] train_0001.jpg | PRED=a GT=a ✅ | ACC=1/1=1.0000 | 2.58s
[0002/1000] train_0002.jpg | PRED=a GT=a ✅ | ACC=2/2=1.0000 | 1.23s
[0003/1000] train_0003.jpg | PRED=d GT=d ✅ | ACC=3/3=1.0000 | 1.22s
[0004/1000] train_0004.jpg | PRED=a GT=a ✅ | ACC=4/4=1.0000 | 1.24s
[0005/1000] train_0006.jpg | PRED=b GT=b ✅ | ACC=5/5=1.0000 | 1.24s
[0006/1000] train_0007.jpg | PRED=b GT=b ✅ | ACC=6/6=1.0000 | 1.23s
[0007/1000] train_0008.jpg | PRED=d GT=d ✅ | ACC=7/7=1.0000 | 1.24s
[0008/1000] train_0009.jpg | PRED=b GT=b ✅ | ACC=8/8=1.0000 | 1.23s
[0009/1000] train_0010.jpg | PRED=d GT=d ✅ | ACC=9/9=1.0000 | 1.24s
[0010/1000] train_0011.jpg | PRED=b GT=b ✅ | ACC=10/10=1.0000 | 1.22s
[0011/1000] train_0012.jpg | PRED=c GT=c ✅ | ACC=11/11=1.0000 | 1.22s
[0012/1000] train_0013.jpg | PRED=c GT=c ✅ | ACC=12/12=1.0000 | 1.23s
[0013/1000] train_0014.jpg | P

# Step 3. Test 전체 추론 및 Submission 생성

Step 2에서 검증한 Qwen3.5-122B Image-only 설정을 그대로 사용하여
대회 Test 데이터 전체를 추론한다.

검증 단계에서 사용한 다음 설정은 변경하지 않는다.

- 실제 이미지 + 질문 + 선택지 a / b / c / d
- OCR 사용하지 않음
- Object Detection 사용하지 않음
- enable_thinking=False
- temperature=0.0
- structured output: a / b / c / d
- max_tokens=2

Test 추론 결과는 중간 저장하여 런타임 오류가 발생하더라도
완료된 문제부터 이어서 실행할 수 있도록 한다.

전체 추론 완료 후 대회 제출 형식인

- id
- answer

두 컬럼으로 Submission CSV를 생성한다.

In [10]:
# ============================================================
# [STEP 3-0] TEST DATA VERIFY
#
# 추론 전에:
# - test.csv 확인
# - sample_submission.csv 확인
# - 필수 컬럼 확인
# - 모든 test 이미지 존재 확인
#
# 아직 모델 추론은 수행하지 않음
# ============================================================

from pathlib import Path

import pandas as pd


# ============================================================
# 0. MODEL / STEP 2 OBJECT CHECK
# ============================================================

assert (
    "llm_122b" in globals()
    and llm_122b is not None
), "❌ Step 1 모델 로드부터 필요합니다."


assert "DATA_ROOT" in globals(), (
    "❌ DATA_ROOT가 없습니다."
)


assert "SYSTEM_PROMPT" in globals(), (
    "❌ Step 2 SYSTEM_PROMPT가 없습니다."
)


assert "build_image_only_prompt" in globals(), (
    "❌ Step 2 build_image_only_prompt가 없습니다."
)


assert "sampling_params_image_only" in globals(), (
    "❌ Step 2 sampling_params_image_only가 없습니다."
)


# ============================================================
# 1. PATH
# ============================================================

TEST_CSV = (
    DATA_ROOT
    / "test.csv"
)

SAMPLE_SUBMISSION_CSV = (
    DATA_ROOT
    / "sample_submission.csv"
)


assert TEST_CSV.exists(), (
    f"❌ test.csv 없음: {TEST_CSV}"
)

assert SAMPLE_SUBMISSION_CSV.exists(), (
    f"❌ sample_submission.csv 없음: {SAMPLE_SUBMISSION_CSV}"
)


# ============================================================
# 2. LOAD
# ============================================================

test_df = pd.read_csv(
    TEST_CSV
)

sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_CSV
)


# ============================================================
# 3. COLUMN CHECK
# ============================================================

required_test_cols = [
    "id",
    "path",
    "question",
    "a",
    "b",
    "c",
    "d",
]


missing = [
    c
    for c in required_test_cols
    if c not in test_df.columns
]


assert not missing, (
    f"❌ test.csv 컬럼 누락: {missing}"
)


assert "id" in sample_submission.columns, (
    "❌ sample_submission.csv에 id 컬럼 없음"
)


assert "answer" in sample_submission.columns, (
    "❌ sample_submission.csv에 answer 컬럼 없음"
)


# ============================================================
# 4. ID / ROW CHECK
# ============================================================

assert len(test_df) == len(sample_submission), (
    "❌ test / sample_submission 행 수 불일치: "
    f"{len(test_df)} vs {len(sample_submission)}"
)


assert (
    test_df["id"]
    .astype(str)
    .tolist()
    ==
    sample_submission["id"]
    .astype(str)
    .tolist()
), "❌ test와 sample_submission의 id 순서가 다릅니다."


# ============================================================
# 5. IMAGE CHECK
# ============================================================

image_exists = test_df["path"].map(
    lambda p:
    (DATA_ROOT / str(p)).exists()
)


missing_images = test_df.loc[
    ~image_exists,
    [
        "id",
        "path",
    ]
]


# ============================================================
# 6. RESULT
# ============================================================

print("=" * 78)
print("STEP 3-0 : TEST DATA VERIFY")
print("=" * 78)

print(
    "DATA_ROOT          :",
    DATA_ROOT
)

print(
    "TEST CSV           :",
    TEST_CSV
)

print(
    "TEST ROWS          :",
    len(test_df)
)

print(
    "TEST COLUMNS       :",
    test_df.columns.tolist()
)

print(
    "SUBMISSION ROWS    :",
    len(sample_submission)
)

print(
    "SUBMISSION COLUMNS :",
    sample_submission.columns.tolist()
)

print(
    "IMAGES             :",
    int(image_exists.sum()),
    "/",
    len(test_df)
)

print("=" * 78)


if len(missing_images) > 0:

    print("\n❌ 누락 이미지:")
    display(
        missing_images.head(20)
    )

else:

    print(
        "\n✅ TEST DATA READY"
    )


display(
    test_df.head(3)
)

display(
    sample_submission.head(3)
)

STEP 3-0 : TEST DATA VERIFY
DATA_ROOT          : /content/ssafy_data
TEST CSV           : /content/ssafy_data/test.csv
TEST ROWS          : 6714
TEST COLUMNS       : ['id', 'path', 'question', 'a', 'b', 'c', 'd']
SUBMISSION ROWS    : 6714
SUBMISSION COLUMNS : ['id', 'answer']
IMAGES             : 6714 / 6714

✅ TEST DATA READY


,id,path,question,a,b,c,d
0,test_0001.jpg,test/test_0001.jpg,이 사진 속 간판 중에서 '머리염색 체험방'이 적혀 있는 간판의 전화번호는 무엇인가요?,010-1234-5678,722-7510,123-4567,732-3258
1,test_0002.jpg,test/test_0002.jpg,이 식당 간판에 적힌 전문점의 종류는 무엇인가요?,중식 전문점,해산물 전문점,소고기숯불구이 전문점,한식 전문점
2,test_0003.jpg,test/test_0003.jpg,이 사진에 보이는 IT대학 발전기금 현황판에서 IT대학 50주년 후원금 중 10억원...,SK hynix,LG,삼성전자,세 가지 모두


,id,answer
0,test_0001.jpg,NaN
1,test_0002.jpg,NaN
2,test_0003.jpg,NaN


In [11]:
# ============================================================
# [STEP 3-1] 122B IMAGE-ONLY TEST FULL INFERENCE
#
# Step 2에서 98.4%를 기록한 설정 그대로 사용
#
# 입력:
# - 실제 이미지
# - question
# - a / b / c / d
#
# 사용하지 않음:
# - OCR
# - DINO
# - GT
#
# 설정:
# - enable_thinking=False
# - temperature=0.0
# - structured output: a/b/c/d
# - max_tokens=2
#
# 기능:
# - 6714개 전체 Test 추론
# - 중간 저장
# - 런타임 중단 후 재실행 시 자동 Resume
# ============================================================

import os
import time
import inspect
import pandas as pd

from pathlib import Path
from PIL import Image


# ============================================================
# 0. CONFIG
# ============================================================

TEST_N = len(test_df)

TEST_OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "cleaning_data/"
    "122b_eval"
)

TEST_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TEST_OUT_CSV = (
    TEST_OUT_DIR
    / "122b_image_only_test.csv"
)


# ============================================================
# 1. OBJECT CHECK
# ============================================================

assert (
    "llm_122b" in globals()
    and llm_122b is not None
), "❌ llm_122b가 없습니다."


assert "SYSTEM_PROMPT" in globals(), (
    "❌ SYSTEM_PROMPT가 없습니다."
)


assert "build_image_only_prompt" in globals(), (
    "❌ build_image_only_prompt가 없습니다."
)


assert "sampling_params_image_only" in globals(), (
    "❌ sampling_params_image_only가 없습니다."
)


assert len(test_df) == 6714, (
    f"❌ 예상 Test 6714개, 실제 {len(test_df)}개"
)


print("=" * 78)
print("122B IMAGE-ONLY TEST FULL INFERENCE")
print("=" * 78)

print(
    "TEST SAMPLES :",
    TEST_N
)

print(
    "OUTPUT       :",
    TEST_OUT_CSV
)


# ============================================================
# 2. RESUME
# ============================================================

test_results = {}


if TEST_OUT_CSV.exists():

    old = pd.read_csv(
        TEST_OUT_CSV
    )

    print(
        "\n기존 결과:",
        len(old),
        "rows"
    )


    for rec in (
        old
        .drop_duplicates(
            "id",
            keep="last"
        )
        .to_dict("records")
    ):

        test_results[
            str(rec["id"])
        ] = rec


test_done_ids = {

    k

    for k, v
    in test_results.items()

    if str(
        v.get(
            "status",
            ""
        )
    ) == "ok"

    and str(
        v.get(
            "image_only_answer",
            ""
        )
    ).strip().lower()

    in {
        "a",
        "b",
        "c",
        "d",
    }
}


print(
    "RESUME       :",
    len(test_done_ids),
    "/",
    TEST_N
)


# ============================================================
# 3. SAVE FUNCTION
# ============================================================

def save_test_results():

    x = pd.DataFrame(
        list(
            test_results.values()
        )
    )


    if len(x) > 0:

        x = (
            x
            .sort_values("idx")
            .reset_index(drop=True)
        )


    tmp = Path(
        str(TEST_OUT_CSV)
        + ".tmp"
    )


    x.to_csv(
        tmp,
        index=False,
        encoding="utf-8-sig",
    )


    os.replace(
        tmp,
        TEST_OUT_CSV,
    )


# ============================================================
# 4. TQDM SUPPORT
# ============================================================

SUPPORTS_USE_TQDM = (
    "use_tqdm"
    in inspect.signature(
        llm_122b.chat
    ).parameters
)


# ============================================================
# 5. LOOP
# ============================================================

run_start = time.time()


for idx, row in test_df.iterrows():

    img_id = str(
        row["id"]
    )


    # 이미 정상 완료된 문제는 Skip
    if img_id in test_done_ids:
        continue


    image_path = (
        DATA_ROOT
        / str(
            row["path"]
        )
    )


    t0 = time.time()


    try:

        assert image_path.exists(), (
            f"이미지 없음: {image_path}"
        )


        with Image.open(
            image_path
        ) as im:

            image = (
                im
                .convert("RGB")
                .copy()
            )


        messages = [

            {
                "role":
                    "system",

                "content":
                    SYSTEM_PROMPT,
            },

            {
                "role":
                    "user",

                "content": [

                    {
                        "type":
                            "image_pil",

                        "image_pil":
                            image,
                    },

                    {
                        "type":
                            "text",

                        "text":
                            build_image_only_prompt(

                                row["question"],

                                row["a"],
                                row["b"],
                                row["c"],
                                row["d"],
                            ),
                    },
                ],
            },
        ]


        kwargs = dict(

            messages=
                messages,

            sampling_params=
                sampling_params_image_only,

            chat_template_kwargs={
                "enable_thinking":
                    False
            },
        )


        if SUPPORTS_USE_TQDM:

            kwargs[
                "use_tqdm"
            ] = False


        outputs = llm_122b.chat(
            **kwargs
        )


        pred = (
            outputs[0]
            .outputs[0]
            .text
            .strip()
            .lower()
        )


        assert pred in {
            "a",
            "b",
            "c",
            "d",
        }, (
            f"비정상 prediction: {repr(pred)}"
        )


        status = "ok"
        error = ""


    except Exception as e:

        pred = ""
        status = "error"
        error = repr(e)


    elapsed = (
        time.time()
        - t0
    )


    # ========================================================
    # RESULT
    # ========================================================

    test_results[
        img_id
    ] = {

        "idx":
            int(idx),

        "id":
            img_id,

        "path":
            row["path"],

        "image_only_answer":
            pred,

        "elapsed_sec":
            round(
                elapsed,
                4
            ),

        "status":
            status,

        "error":
            error,
    }


    if status == "ok":

        test_done_ids.add(
            img_id
        )


    # 매 문제 저장
    save_test_results()


    # ========================================================
    # PROGRESS
    # ========================================================

    done_n = len(
        test_done_ids
    )


    if status == "ok":

        mark = "✅"

    else:

        mark = "❌"


    print(

        f"[{done_n:04d}/{TEST_N}] "

        f"{img_id} | "

        f"PRED={pred} "

        f"{mark} | "

        f"{elapsed:.2f}s",

        flush=True,
    )


# ============================================================
# 6. FINAL CHECK
# ============================================================

final_test = pd.DataFrame(
    list(
        test_results.values()
    )
)


final_test = (
    final_test
    .sort_values("idx")
    .reset_index(drop=True)
)


ok_test = final_test[
    final_test["status"]
    == "ok"
]


error_test = final_test[
    final_test["status"]
    != "ok"
]


valid_answer = (
    ok_test[
        "image_only_answer"
    ]
    .astype(str)
    .str.lower()
    .isin(
        {
            "a",
            "b",
            "c",
            "d",
        }
    )
)


print("\n" + "=" * 78)

print(
    "122B TEST INFERENCE FINISHED"
)

print("=" * 78)

print(
    "TOTAL SAVED :",
    len(final_test)
)

print(
    "OK          :",
    len(ok_test)
)

print(
    "ERROR       :",
    len(error_test)
)

print(
    "VALID ANSWER:",
    int(valid_answer.sum()),
    "/",
    len(ok_test)
)

print(
    "RESULT      :",
    TEST_OUT_CSV
)

print("=" * 78)


if len(error_test) > 0:

    print(
        "\n⚠️ ERROR CASES"
    )

    display(
        error_test[
            [
                "idx",
                "id",
                "path",
                "error",
            ]
        ]
    )

else:

    print(
        "\n✅ ALL 6714 TEST SAMPLES COMPLETE"
    )


print(
    "\nANSWER DISTRIBUTION:"
)

print(
    ok_test[
        "image_only_answer"
    ]
    .value_counts()
    .sort_index()
)

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
[1727/6714] test_1727.jpg | PRED=a ✅ | 1.23s
[1728/6714] test_1728.jpg | PRED=c ✅ | 1.22s
[1729/6714] test_1729.jpg | PRED=b ✅ | 1.32s
[1730/6714] test_1730.jpg | PRED=d ✅ | 1.21s
[1731/6714] test_1731.jpg | PRED=c ✅ | 1.23s
[1732/6714] test_1732.jpg | PRED=c ✅ | 1.22s
[1733/6714] test_1733.jpg | PRED=d ✅ | 1.23s
[1734/6714] test_1734.jpg | PRED=b ✅ | 1.22s
[1735/6714] test_1735.jpg | PRED=a ✅ | 1.23s
[1736/6714] test_1736.jpg | PRED=c ✅ | 1.33s
[1737/6714] test_1737.jpg | PRED=b ✅ | 1.24s
[1738/6714] test_1738.jpg | PRED=c ✅ | 1.22s
[1739/6714] test_1739.jpg | PRED=c ✅ | 1.22s
[1740/6714] test_1740.jpg | PRED=b ✅ | 1.22s
[1741/6714] test_1741.jpg | PRED=a ✅ | 1.23s
[1742/6714] test_1742.jpg | PRED=d ✅ | 1.23s
[1743/6714] test_1743.jpg | PRED=b ✅ | 1.23s
[1744/6714] test_1744.jpg | PRED=c ✅ | 1.23s
[1745/6714] test_1745.jpg | PRED=b ✅ | 1.22s
[1746/6714] test_1746.jpg | PRED=d ✅ | 1.21s
[1747/6714] test_1747.jpg | PRED=d ✅ | 1.23s
[1748/6714] test_17

,idx,id,path,error
5985,5985,test_5986.jpg,test/test_5986.jpg,VLLMValidationError('The decoder prompt (lengt...
6686,6686,test_6687.jpg,test/test_6687.jpg,VLLMValidationError('The decoder prompt (lengt...



ANSWER DISTRIBUTION:
image_only_answer
a    1700
b    1678
c    1650
d    1684
Name: count, dtype: int64


## Step 3-2. 오류 샘플 재추론 및 Submission 생성

전체 Test 6,714개 중 6,712개가 정상적으로 추론되었으며,
2개 샘플에서 입력 길이가 `max_model_len=2048`을 초과하여 오류가 발생했다.

오류가 발생한 2개 이미지만 해상도를 축소하여 다시 추론하고,
모든 Test 샘플의 예측값이 a / b / c / d 중 하나인지 최종 검증한다.

이후 `sample_submission.csv`의 id 순서를 그대로 유지하여
최종 제출 파일을 생성한다.

최종 제출 형식:

- id
- answer

In [12]:
# ============================================================
# [STEP 3-2] RETRY 2 ERROR CASES + MAKE SUBMISSION
# ============================================================

import os
import time
import inspect
import pandas as pd

from pathlib import Path
from PIL import Image


# ============================================================
# 0. PATH
# ============================================================

TEST_OUT_CSV = Path(
    "/content/drive/MyDrive/cleaning_data/122b_eval/122b_image_only_test.csv"
)

SUBMISSION_OUT = Path(
    "/content/drive/MyDrive/cleaning_data/122b_eval/122b_submission.csv"
)


# ============================================================
# 1. LOAD CURRENT RESULT
# ============================================================

result_df = pd.read_csv(
    TEST_OUT_CSV
)

error_df = result_df[
    result_df["status"] != "ok"
].copy()


print("=" * 78)
print("ERROR RETRY")
print("=" * 78)

print(
    "ERROR CASES:",
    len(error_df)
)

display(
    error_df[
        [
            "idx",
            "id",
            "path",
            "error",
        ]
    ]
)


# ============================================================
# 2. RETRY ONLY ERROR CASES
#
# 입력 길이 초과 문제이므로
# 이미지 해상도만 줄여 visual token 수 감소
# ============================================================

SUPPORTS_USE_TQDM = (
    "use_tqdm"
    in inspect.signature(
        llm_122b.chat
    ).parameters
)


for _, err_row in error_df.iterrows():

    idx = int(
        err_row["idx"]
    )

    row = test_df.iloc[
        idx
    ]

    image_path = (
        DATA_ROOT
        / str(
            row["path"]
        )
    )

    print(
        "\nRETRY:",
        idx,
        row["id"]
    )


    t0 = time.time()


    try:

        with Image.open(
            image_path
        ) as im:

            image = im.convert(
                "RGB"
            )

            # 원본 비율 유지
            # 긴 변을 최대 1024px로 축소
            image.thumbnail(
                (1024, 1024),
                Image.Resampling.LANCZOS
            )

            image = image.copy()


        print(
            "resized image:",
            image.size
        )


        messages = [

            {
                "role":
                    "system",

                "content":
                    SYSTEM_PROMPT,
            },

            {
                "role":
                    "user",

                "content": [

                    {
                        "type":
                            "image_pil",

                        "image_pil":
                            image,
                    },

                    {
                        "type":
                            "text",

                        "text":
                            build_image_only_prompt(

                                row["question"],

                                row["a"],
                                row["b"],
                                row["c"],
                                row["d"],
                            ),
                    },
                ],
            },
        ]


        kwargs = dict(

            messages=
                messages,

            sampling_params=
                sampling_params_image_only,

            chat_template_kwargs={
                "enable_thinking":
                    False
            },
        )


        if SUPPORTS_USE_TQDM:

            kwargs[
                "use_tqdm"
            ] = False


        outputs = llm_122b.chat(
            **kwargs
        )


        pred = (
            outputs[0]
            .outputs[0]
            .text
            .strip()
            .lower()
        )


        assert pred in {
            "a",
            "b",
            "c",
            "d",
        }


        elapsed = (
            time.time()
            - t0
        )


        # 기존 row 수정
        result_df.loc[
            result_df["idx"] == idx,
            "image_only_answer"
        ] = pred

        result_df.loc[
            result_df["idx"] == idx,
            "elapsed_sec"
        ] = round(
            elapsed,
            4
        )

        result_df.loc[
            result_df["idx"] == idx,
            "status"
        ] = "ok"

        result_df.loc[
            result_df["idx"] == idx,
            "error"
        ] = ""


        print(
            f"✅ {row['id']} -> {pred} "
            f"({elapsed:.2f}s)"
        )


    except Exception as e:

        print(
            "❌ RETRY FAILED:",
            repr(e)
        )


# ============================================================
# 3. SAVE UPDATED FULL RESULT
# ============================================================

result_df = (
    result_df
    .sort_values("idx")
    .reset_index(drop=True)
)

result_df.to_csv(
    TEST_OUT_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 4. FINAL VALIDATION
# ============================================================

valid_mask = (
    result_df["status"].eq("ok")
    &
    result_df[
        "image_only_answer"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(
        ["a", "b", "c", "d"]
    )
)


print("\n" + "=" * 78)
print("FINAL TEST RESULT")
print("=" * 78)

print(
    "TOTAL:",
    len(result_df)
)

print(
    "VALID:",
    int(valid_mask.sum())
)

print(
    "ERROR:",
    int((~valid_mask).sum())
)


assert len(result_df) == len(test_df) == 6714

assert valid_mask.all(), (
    "❌ 아직 정상 답변이 없는 Test 샘플이 있습니다."
)


# ============================================================
# 5. MAKE SUBMISSION
#
# sample_submission의 id 순서를 그대로 사용
# ============================================================

pred_map = dict(
    zip(
        result_df["id"].astype(str),
        result_df[
            "image_only_answer"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
    )
)


submission = (
    sample_submission[
        ["id"]
    ]
    .copy()
)


submission[
    "answer"
] = (
    submission["id"]
    .astype(str)
    .map(
        pred_map
    )
)


# ============================================================
# 6. SUBMISSION VALIDATION
# ============================================================

assert len(submission) == 6714

assert submission[
    "answer"
].notna().all()

assert submission[
    "answer"
].isin(
    ["a", "b", "c", "d"]
).all()

assert (
    submission["id"]
    .astype(str)
    .tolist()
    ==
    sample_submission["id"]
    .astype(str)
    .tolist()
)


# ============================================================
# 7. SAVE
# ============================================================

submission.to_csv(
    SUBMISSION_OUT,
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 78)
print("SUBMISSION READY")
print("=" * 78)

print(
    "ROWS :",
    len(submission)
)

print(
    "PATH :",
    SUBMISSION_OUT
)

print("\nANSWER DISTRIBUTION:")

print(
    submission[
        "answer"
    ]
    .value_counts()
    .sort_index()
)

print("=" * 78)

display(
    submission.head()
)

ERROR RETRY
ERROR CASES: 2


,idx,id,path,error
5985,5985,test_5986.jpg,test/test_5986.jpg,VLLMValidationError('The decoder prompt (lengt...
6686,6686,test_6687.jpg,test/test_6687.jpg,VLLMValidationError('The decoder prompt (lengt...



RETRY: 5985 test_5986.jpg
resized image: (238, 1024)
✅ test_5986.jpg -> b (1.12s)

RETRY: 6686 test_6687.jpg
resized image: (1024, 217)
✅ test_6687.jpg -> d (1.07s)

FINAL TEST RESULT
TOTAL: 6714
VALID: 6714
ERROR: 0

SUBMISSION READY
ROWS : 6714
PATH : /content/drive/MyDrive/cleaning_data/122b_eval/122b_submission.csv

ANSWER DISTRIBUTION:
answer
a    1700
b    1679
c    1650
d    1685
Name: count, dtype: int64


,id,answer
0,test_0001.jpg,b
1,test_0002.jpg,c
2,test_0003.jpg,d
3,test_0004.jpg,d
4,test_0005.jpg,c
